# Phase C — does removing the GAN stop the collapse?The with-GAN run collapsed **monotonically**: recall 0.104 → 0.003 and FID 122 → 175between steps 5000 and 20000 (LOG ENTRY 014). Two candidate causes were left open:* **(A) the critic lags** — `d_steps=1` is one critic update per generator update, i.e.  *not* the two-timescale rule the code comment claims.* **(B) the discriminator is the problem** — the cheap tier never runs DMD2's real  critic-feature head; it substitutes a 3.0M standalone conv net on raw pixels.This notebook scores the `gan_weight=0` run, which holds everything else fixed andremoves the discriminator entirely. **Collapse gone → (B). Collapse persists → (A).**It also scores the *teacher* through the identical metric, because nothing has yetestablished what this precision/recall returns for a model known to be good.

In [ ]:
#@title 1 · Setup!git clone -q -b worktree-runbook https://github.com/pakhomovee/diffusion-distill.git%cd diffusion-distill!pip install -q diffusers pytorch-fid pyarrowimport torch, subprocessprint(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
#@title 2 · What to scoreREPO     = "pakhomovee/distill"       #@param {type:"string"}RUN      = "cifar10_dmd2_s40_nogan"   #@param {type:"string"}OUT      = "out"                      #@param {type:"string"}# n=5000 on purpose: the with-GAN trajectory below was scored at n=5000, and FID# depends on the sample count. Comparing across different n compares nothing.FID_N    = 5000TEACHER_STEPS = 18                    # Heun; NFE ~ 2x thisTEACHER_N     = FID_N                 # same n, or recall is not comparable either# LOG ENTRY 014, cifar10_dmd2_s40 (sigma_max=40, gan_weight=1e-3), n=5000.WITH_GAN = [(5000, 121.6, 0.614, 0.1044), (10000, 136.0, 0.628, 0.0834),            (15000, 165.2, 0.644, 0.0414), (20000, 175.1, 0.682, 0.0028)]

In [ ]:
#@title 3 · Fetch the checkpointsimport os, refrom huggingface_hub import list_repo_files, hf_hub_downloadfiles = [f for f in list_repo_files(REPO, repo_type="dataset")         if f.startswith(RUN + "/") and re.search(r"ckpt_.*\.pt$", f)]key = lambda f: (os.path.basename(f)[5:-3] == "final",                 int(m.group()) if (m := re.search(r"\d+", os.path.basename(f))) else -1)files = sorted(files, key=key)assert files, f"no ckpt_*.pt under {RUN}/ in {REPO} -- is the upload finished?"local = [hf_hub_download(REPO, f, repo_type="dataset") for f in files]for f, p in zip(files, local):    print(f"{f:50} {os.path.getsize(p)/1e6:7.1f} MB")

In [ ]:
#@title 4 · Score every checkpoint  (builds the FID reference once, then caches it)import subprocess, jsonfor p in local:    subprocess.run(["python3", "scripts/colab_check.py", "--ckpt", p,                    "--fid-n", str(FID_N), "--out-dir", OUT], check=True)recs = json.load(open(f"{OUT}/scores.json"))print(f"\n{'step':>7} {'FID':>8} {'prec':>7} {'recall':>8}")for r in recs:    print(f"{r['step']:>7} {r['fid']:8.1f} {r['prec']:7.3f} {r['recall']:8.4f}")

In [ ]:
#@title 5 · The ceiling — score the TEACHER through the same metric# If the teacher's recall is also ~0, the metric is what is broken, not the student.REF = f"{OUT}/ref_32_50000.npz"     # cached by cell 4assert os.path.exists(REF), "run cell 4 first -- it builds the reference"subprocess.run(["python3", "scripts/teacher_sample.py",                "--teacher", "diffusers:google/ddpm-cifar10-32",                "--steps", str(TEACHER_STEPS), "--n", "64",                "--fid-n", str(TEACHER_N), "--batch", "64",                "--ref", REF, "--out", f"{OUT}/teacher.png"], check=True)

In [ ]:
#@title 6 · Trajectories, side by sideimport matplotlib.pyplot as pltng = [(r["step"], r["fid"], r["prec"], r["recall"]) for r in recs]fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))for name, tr, style in (("with GAN (1e-3)", WITH_GAN, "o--"), ("no GAN", ng, "o-")):    s = [t[0] for t in tr]    ax[0].plot(s, [t[1] for t in tr], style, label=name)    ax[1].plot(s, [t[2] for t in tr], style, label=name)    ax[2].plot(s, [t[3] for t in tr], style, label=name)for a, t in zip(ax, ("FID (lower better)", "precision", "recall (coverage)")):    a.set_title(t); a.set_xlabel("step"); a.grid(alpha=.3)ax[2].axhline(0, color="k", lw=.5)ax[0].legend()plt.tight_layout(); plt.show()

In [ ]:
#@title 7 · Verdictbest = min(recs, key=lambda r: r["fid"])peak = max(recs, key=lambda r: r["recall"])last, first = recs[-1], recs[0]drop = (peak["recall"] - last["recall"]) / max(peak["recall"], 1e-9)print(f"best FID      : {best['fid']:.1f} at step {best['step']}")print(f"peak recall   : {peak['recall']:.4f} at step {peak['step']}")print(f"final recall  : {last['recall']:.4f}  ({drop:+.0%} from peak)")print()if drop > 0.5 and peak["step"] < last["step"]:    print("COLLAPSES ANYWAY -> the discriminator is NOT the cause.")    print("  Hypothesis (A): the critic lags. Next probe: --set d_steps=5.")elif last["step"] == peak["step"] or drop < 0.2:    print("NO COLLAPSE -> the ConvGANHead substitution is implicated.")    print("  The baseline is not DMD2, and 'Track A beats DMD2' does not follow")    print("  until the real critic-feature head is used or the claim is narrowed.")else:    print("AMBIGUOUS -- recall moved but not decisively. Score more checkpoints,")    print("or extend training, before concluding either way.")print(f"\nBest checkpoint is step {best['step']}; eval_all.sh would have scored 'final'.")

In [ ]:
#@title 8 · Look at themfrom IPython.display import Image, displayimport globprint("TEACHER (the ceiling)"); display(Image(f"{OUT}/teacher.png"))for p in sorted(glob.glob(f"{OUT}/samples_*.png")):    if "guard_off" in p: continue    print(p); display(Image(p))subprocess.run(["python3", "scripts/sample_stats.py",                f"{OUT}/samples_{best['step']}.png"], check=False)